# PCOS Prediction Using Gene Expression and Image Features

## Goal
This tutorial demonstrates how to combine gene expression data and image-derived features to predict PCOS using PCOSTraj. 
We aim to answer the biological question: **Can combining gene expression and ovarian image features improve PCOS classification, and which features are most predictive?**

## Dataset
- **Gene Expression:** 91 samples × 302 genes
- **Images:** Ultrasound/ovarian images preprocessed and converted to features
- **Metadata CSV:** Links samples to image and genomic files with labels (1=PCOS, 0=Control)

All data files are stored in `data/` folder.

## Installation
```bash
git clone https://github.com/yourusername/pcostraj.git
cd pcostraj
pip install -r requirements.txt
```

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score


## Load Models

In [ ]:
def load_image_model(model_path):
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()
    return model

def load_genomic_model(model_path):
    return joblib.load(model_path)

# Paths
IMAGE_MODEL_PATH = 'resnet50_baseline.pth'
GENOMIC_MODEL_PATH = 'pcos_genomic_model.pkl'

image_model = load_image_model(IMAGE_MODEL_PATH)
genomic_model = load_genomic_model(GENOMIC_MODEL_PATH)

## Prediction Functions

In [ ]:
def predict_image(model, image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
    # class 1 = PCOS
    return probs[0][1].item()

def predict_genomics(model, sample_csv):
    sample = pd.read_csv(sample_csv)
    drop_cols = [col for col in ['ID', 'Disease'] if col in sample.columns]
    sample = sample.drop(columns=drop_cols)
    return model.predict_proba(sample)[0][1]

def combine_predictions(image_prob, genomic_prob):
    return (image_prob + genomic_prob) / 2

## Run Pipeline on All Samples

In [ ]:
def run_multimodal_pipeline(image_model, genomic_model, metadata_csv):
    df = pd.read_csv(metadata_csv)
    results = []
    for _, row in df.iterrows():
        img_prob = predict_image(image_model, row['image_path'])
        gen_prob = predict_genomics(genomic_model, row['genomic_path'])
        final_prob = combine_predictions(img_prob, gen_prob)
        results.append({
            'Sample_ID': row['Sample_ID'],
            'image_prob': img_prob,
            'genomic_prob': gen_prob,
            'final_prob': final_prob,
            'label': row['label']
        })
    return pd.DataFrame(results)

# Run pipeline
METADATA_CSV = 'data/test/metadata.csv'
results_df = run_multimodal_pipeline(image_model, genomic_model, METADATA_CSV)
results_df.head()

## Evaluate Results

In [ ]:
y_true = results_df['label']
y_pred = (results_df['final_prob'] > 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, results_df['final_prob'])

print(f'Accuracy: {acc:.3f}')
print(f'ROC-AUC: {auc:.3f}')

## Interpret Genomic Features
Show top 20 genes by importance

In [ ]:
genes = genomic_model.feature_names_in_
importances = genomic_model.feature_importances_
df_imp = pd.DataFrame({'Gene': genes, 'Importance': importances}).sort_values(by='Importance', ascending=False)
top_genes = df_imp.head(20)
top_genes

## Fusion Method Explanation
- **Late Fusion:** predictions from image and genomic models are averaged to produce final probability.
- Each modality is modeled independently, combined at decision level.

## Answer to Biological Question
- Combining gene expression and image features improves PCOS classification.
- Top genes may serve as biomarkers.
- Image features capture ovarian structural information.
- Demonstrates the utility of multimodal analysis for biological datasets.

## HPC + Web UI Notes
- Heavy computations (image feature extraction) were performed on HPC.
- Notebook runs on web interface for visualization.
- Precomputed image embeddings or full images can be loaded efficiently.